# New Zi init

In [39]:
from math import sqrt
from numbers import Complex

class Zi(Complex):

    __slots__ = ('_real', '_imag')

    def __init__(self, real: (int, float, complex) = None, imag: (int, float) = None):

        if isinstance(real, (complex, Zi)):
            if imag is None:
                super().__setattr__('_real', round(real.real))
                super().__setattr__('_imag', round(real.imag))
            else:
                raise TypeError(f"imag must be None if real is a complex: {imag}")

        elif isinstance(real, (int, float)):
            super().__setattr__('_real', round(real))

            if isinstance(imag, (int, float)):
                super().__setattr__('_imag', round(imag))

            elif imag is None:
                super().__setattr__('_imag', 0)

            else:
                raise TypeError(f"Invalid type for imag: {imag}")

        elif real is None and imag is None:
            super().__setattr__('_real', 0)
            super().__setattr__('_imag', 0)
        
        else:
            raise TypeError(f"Invalid type for real: {real}")

    @property
    def real(self) -> int:
        return self._real

    @property
    def imag(self) -> int:
        return self._imag

    @staticmethod
    def _ensure_zi(x):
        if isinstance(x, Zi):
            return x
        if isinstance(x, complex):
            return Zi(x)
        if isinstance(x, (int, float)):
            return Zi(x, 0)
        raise TypeError(f"Cannot convert {type(x)} to Zi")

    def __getitem__(self, idx):
        if idx == 0:
            return self.real
        elif idx == 1:
            return self.imag
        raise IndexError("Zi index out of range (must be 0 or 1)")

    # def __eq__(self, other):
    #     oth = Zi._ensure_zi(other)
    #     return self.real == oth.real and self.imag == oth.imag

    def __eq__(self, other):
        try:
            oth = Zi._ensure_zi(other)
        except TypeError:
            return NotImplemented
        return self.real == oth.real and self.imag == oth.imag

    #---------------- UNIVARIATE METHODS -----------------------
        
    def __repr__(self):
        return f"Zi({self.real}, {self.imag})"

    def __str__(self):
        if self.imag == 0:
            return str(self.real)
        return str(complex(self.real, self.imag))

    def __hash__(self):
        return hash((self.real, self.imag))

    def __complex__(self):
        return complex(self.real, self.imag)
        
    def __abs__(self):
        return sqrt(self.norm())

    def __neg__(self):
        return Zi(-self.real, -self.imag)
        
    def __pos__(self):
        return Zi(self.real, self.imag)
        
    def conjugate(self):
        return Zi(self.real, -self.imag)

    def norm(self):
        return self.real * self.real + self.imag * self.imag

    #---------------- ARITHMETIC -----------------------------
        
    def __add__(self, other):
        oth = Zi._ensure_zi(other)
        return Zi(self.real + oth.real, self.imag + oth.imag)
        
    def __radd__(self, other):
        oth = Zi._ensure_zi(other)
        return oth + self

    def __iadd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        oth = Zi._ensure_zi(other)
        return Zi(self.real - oth.real, self.imag - oth.imag)
        
    def __mul__(self, other):
        oth = Zi._ensure_zi(other)
        a, b = self
        c, d = oth
        return Zi(a * c - b * d, a * d + b * c)
        
    def __rmul__(self, other):
        oth = Zi._ensure_zi(other)
        return oth * self          # multiplication is commutative

    def __truediv__(self, other):
        oth = Zi._ensure_zi(other)
        n = oth.norm()
        if n == 0:
            raise ZeroDivisionError("division by zero Gaussian integer")
        num = self * oth.conjugate()          # (a+bi)(c-di) via existing __mul__
        return Zi(round(num.real / n), round(num.imag / n))

    def __rtruediv__(self, other):
        oth = Zi._ensure_zi(other)
        return oth.__truediv__(self)

    def __pow__(self, exponent):
        if not isinstance(exponent, int):
            return NotImplemented
        if exponent == 0:
            return Zi(1, 0)
        base, exp = (self, exponent) if exponent > 0 else (Zi(1, 0) / self, -exponent)
        result = Zi(1, 0)
        while exp > 0:
            if exp & 1:
                result = result * base
            base = base * base
            exp >>= 1
        return result

    def __rpow__(self, base):
        if self.imag != 0:
            return NotImplemented          # exponent must be an ordinary integer
        return Zi._ensure_zi(base).__pow__(self.real)

In [40]:
print(f"{Zi() = }")
print(f"{Zi(0) = }")
print(f"{Zi(1) = }")
print(f"{Zi((1+2j)) = }")
# print(f"{Zi( (1+2j), (3+4j) ) = }")  # Error

Zi() = Zi(0, 0)
Zi(0) = Zi(0, 0)
Zi(1) = Zi(1, 0)
Zi((1+2j)) = Zi(1, 2)


In [41]:
c1 = Zi(4, 5)
c2 = Zi(1, -2)

In [42]:
Zi(4, 5) + 2

Zi(6, 5)

In [43]:
2 + Zi(4, 5)

Zi(6, 5)

In [44]:
print(f"{c1} + {c2} = {c1 + c2}\n")  # Zi + Zi

(4+5j) + (1-2j) = (5+3j)



In [45]:
print(f"{c1} + 2 = {c1 + 2}")  # Zi + int

(4+5j) + 2 = (6+5j)


In [46]:
print(f"2 + {c1} = {2 + c1}\n")

2 + (4+5j) = (6+5j)



In [47]:
print(f"{c1} + 1.9 = {c1 + 1.9}")  # Zi + float
print(f"1.9 + {c1} = {1.9 + c1}\n")  # float + Zi

(4+5j) + 1.9 = (6+5j)
1.9 + (4+5j) = (6+5j)



In [48]:
cx = (1-1j)
print(f"{c1} + {cx} = {c1 + cx}")  # Zi + complex
print(f"{cx} + {c1} = {cx + c1}")  # complex + Zi

(4+5j) + (1-1j) = (5+4j)
(1-1j) + (4+5j) = (5+4j)


In [49]:
2 + (1 + 3j)

(3+3j)

In [50]:
c1 + 2

Zi(6, 5)

In [51]:
2 + c1

Zi(6, 5)

In [52]:
foo = Zi._ensure_zi(2)

In [53]:
type(foo.real)

int

Good news: most of the "plumbing" is right (the `Complex` ABC registration, `__getitem__`-based unpacking, `__hash__`, `__complex__`). But there are several real bugs. Let me verify a few with actual execution rather than just eyeballing:That confirms several real bugs beyond the duplicate `__radd__`. Here's the priority list:

**1. `self._re` doesn't exist — it's `self._real`**

This breaks `__neg__`, `__pos__`, `conjugate`, and `__str__` (when `imag == 0`):

```python
def __neg__(self):
    return Zi(-self._real, -self.imag)   # was self._re

def __pos__(self):
    return Zi(self._real, self.imag)     # was self._re

def conjugate(self):
    return Zi(self._real, -self.imag)    # was self._re

def __str__(self):
    if self.imag == 0:
        return str(self._real)           # was self._re
    return str(complex(self.real, self.imag))
```

Since `Complex.__rsub__` (inherited from the ABC) is defined as `-self + other`, this bug silently breaks `2 - Zi(4,5)` too — that's the `rsub` failure above.

**2. `__rmul__`, `__truediv__`, `__rtruediv__`, `__pow__`, `__rpow__` are broken stubs** — same pattern as the `__radd__` issue. They all `return NotImplemented` unconditionally, and the division/pow ones are even missing the `other` parameter, so calling them directly would raise a `TypeError` about argument count. `2 * Zi(4,5)` fails for the same reason `2 + Zi(4,5)` did.

**3. `__eq__` raises instead of returning `NotImplemented`.** Python's data model expects rich comparisons to return `NotImplemented` for incomparable types, not raise:

```python
def __eq__(self, other):
    try:
        oth = Zi._ensure_zi(other)
    except TypeError:
        return NotImplemented
    return self.real == oth.real and self.imag == oth.imag
```

Without this, `z == "hello"` throws instead of just being `False`, and things like `z in some_list` or `assertEqual` against unrelated types will blow up.

**4. Hash/equality contract violation.** `_ensure_zi` lets `Zi(5, 0) == 5` be `True`, but `hash(Zi(5,0))` (`hash((5,0))`) ≠ `hash(5)`. Python requires equal objects to hash equal — violating this silently corrupts dict/set behavior. Either don't allow cross-type equality with plain numbers, or normalize the hash (e.g., hash as `complex(self)` when `imag == 0` matches a real).

Given you're fuzz-testing, I'd fix #1 and #3 immediately (they're outright crashes), then decide the intended semantics for #2 (should division return `Zi` rounded to nearest Gaussian integer, or should it produce a `Qi`?) before implementing. Want me to write out proper `__mul__`-consistent `__rmul__`, plus `__truediv__`/`__rtruediv__`/`__pow__`/`__rpow__` implementations?

```
def __rmul__(self, other):
    oth = Zi._ensure_zi(other)
    return oth * self          # multiplication is commutative

def __truediv__(self, other):
    oth = Zi._ensure_zi(other)
    n = oth.norm()
    if n == 0:
        raise ZeroDivisionError("division by zero Gaussian integer")
    num = self * oth.conjugate()          # (a+bi)(c-di) via existing __mul__
    return Zi(round(num.real / n), round(num.imag / n))

def __rtruediv__(self, other):
    oth = Zi._ensure_zi(other)
    return oth.__truediv__(self)

def __pow__(self, exponent):
    if not isinstance(exponent, int):
        return NotImplemented
    if exponent == 0:
        return Zi(1, 0)
    base, exp = (self, exponent) if exponent > 0 else (Zi(1, 0) / self, -exponent)
    result = Zi(1, 0)
    while exp > 0:
        if exp & 1:
            result = result * base
        base = base * base
        exp >>= 1
    return result

def __rpow__(self, base):
    if self.imag != 0:
        return NotImplemented          # exponent must be an ordinary integer
    return Zi._ensure_zi(base).__pow__(self.real)
```

Here's a consistent set, following the same pattern you used for `__add__`/`__radd__` (delegate to the primary op) and the standard Gaussian-integer division algorithm (divide, then round to nearest lattice point):

```python
def __rmul__(self, other):
    oth = Zi._ensure_zi(other)
    return oth * self          # multiplication is commutative

def __truediv__(self, other):
    oth = Zi._ensure_zi(other)
    n = oth.norm()
    if n == 0:
        raise ZeroDivisionError("division by zero Gaussian integer")
    num = self * oth.conjugate()          # (a+bi)(c-di) via existing __mul__
    return Zi(round(num.real / n), round(num.imag / n))

def __rtruediv__(self, other):
    oth = Zi._ensure_zi(other)
    return oth.__truediv__(self)

def __pow__(self, exponent):
    if not isinstance(exponent, int):
        return NotImplemented
    if exponent == 0:
        return Zi(1, 0)
    base, exp = (self, exponent) if exponent > 0 else (Zi(1, 0) / self, -exponent)
    result = Zi(1, 0)
    while exp > 0:
        if exp & 1:
            result = result * base
        base = base * base
        exp >>= 1
    return result

def __rpow__(self, base):
    if self.imag != 0:
        return NotImplemented          # exponent must be an ordinary integer
    return Zi._ensure_zi(base).__pow__(self.real)
```

Key design points, since these carry real math decisions rather than just plumbing:

- **`__truediv__` returns a `Zi`, not a `Qi` or `complex`.** Division isn't exact in `ℤ[i]`, so this rounds to the nearest Gaussian integer using the standard `a·b̄ / N(b)` trick, then `round()` on each coordinate (Python's `round()` is banker's-rounding, round-half-to-even). If you actually want exact quotients, that's what your `Qi` class is for — this operator is the "nearest lattice point" version, consistent with `modified_divmod`. Worth documenting clearly in the docstring so nobody's surprised that `Zi(1,0) / Zi(2,0)` gives `Zi(0,0)` or `Zi(1,0)` rather than a fraction.
- **`__pow__` supports negative integer exponents** by first inverting via `__truediv__` (so again, subject to rounding for non-unit bases) — e.g. `Zi(2,0) ** -1` won't be a true inverse since 2 isn't a unit in `ℤ[i]`. Only `±1, ±i` have exact inverses. Might be worth raising instead of silently rounding for negative exponents on non-units — your call depending on what the fuzz tests expect.
- **`__rpow__` only handles the case where the exponent (`self`) has `imag == 0`** — `Zi` exponents with nonzero imaginary part aren't well-defined in this ring, so it correctly bails to `NotImplemented` rather than guessing.

Let me verify these actually work end-to-end:All confirmed working end-to-end. A couple of things visible in that test output worth flagging:

- **`z ** -1 = 0`** — this isn't a bug, it's the expected consequence of rounding: `Zi(4,5)` isn't a unit in `ℤ[i]` (only `±1, ±i` have exact multiplicative inverses), so `1/z` rounds to the nearest lattice point, which happens to be `Zi(0,0)`. If your fuzz tests assert `z * (z**-1) == 1` for arbitrary `z`, that assertion will correctly fail for non-units — which might be exactly the kind of edge case you want the fuzz suite to catch, or might mean you want `__pow__` to raise on negative exponents unless `norm() == 1`. Worth deciding explicitly rather than letting it round silently.
- **`2 ** Zi(3, 0)` correctly falls back to `__rpow__`** and returns `8` — confirms the ABC dispatch chain (`int.__pow__` → `NotImplemented` → `Zi.__rpow__`) works the same way `__radd__` does.

One more thing since you're fuzz-testing heavily: your `__mul__`/`__truediv__` combo should satisfy `(a * b) / b == a` for any nonzero `b` where the division is exact (i.e., when `b | a` in `ℤ[i]`), and that's a good property to explicitly fuzz-test given the rounding behavior — it'll catch any sign or rounding-direction errors in the `a·b̄/N(b)` formula that a spot check like the one above might miss.

Glad it all checks out! Good luck with the rest of the `Gauss`/`Zi`/`Qi` work — feel free to come back when you're ready to fuzz-test `xgcd` or the `modified_divmod` rounding tie-points, since those tend to have trickier edge cases than the basic ring operations.